# 01 — Boundaries: Urban Extent Detection

Test case: **Config-driven country pipeline**

Pipeline:
1. Load boundary polygons + join census population
2. Reproject to metric CRS, calculate density
3. For each city: clip to radius around centroid
4. Filter to dense units
5. Buffer + dissolve into contiguous clusters
6. Re-summarise population/area per cluster
7. Apply city-level filters (pop > 100k, area > 50km²)
8. Visualise + log results

## Setup

In [ ]:
import sys
from pathlib import Path

# Notebook lives in notebooks/, so project root is one level up
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import geopandas as gpd
import matplotlib.pyplot as plt

from walkability.config import load_config
from walkability.boundaries import (
    city_slug,
    load_boundaries_with_population,
    calculate_density,
    clip_to_city,
    filter_dense_units,
    dissolve_clusters,
    normalize_german_city_name,
    summarise_clusters,
    apply_city_filters,
    process_city,
)

In [ ]:
# ---- Loaded from country config ----
COUNTRY = "germany"
config = load_config(COUNTRY, config_dir=str(PROJECT_ROOT / "config"))

BOUNDARY_SHP = PROJECT_ROOT / config.boundary_file
CENSUS_CSV = PROJECT_ROOT / config.census_file
BOUNDARY_JOIN_COL = config.join_key_boundary
CENSUS_JOIN_COL = config.join_key_census
POPULATION_COL = config.population_col
METRIC_CRS = config.crs
TEST_CITY = config.cities[0]
# -----------------------------------

config

## Step 1 — Load boundaries + join population

In [ ]:
gdf = load_boundaries_with_population(
    str(BOUNDARY_SHP),
    str(CENSUS_CSV),
    BOUNDARY_JOIN_COL,
    CENSUS_JOIN_COL,
    POPULATION_COL,
    census_read_csv_options=config.census_read_csv_options,
    census_filters=config.census_filters,
)
print(f"Loaded {len(gdf)} boundary units")
gdf.head()

## Step 2 — Reproject + calculate density

In [ ]:
gdf_metric = calculate_density(gdf, metric_crs=METRIC_CRS)
gdf_metric[["population", "area_km2", "pop_density"]].describe()

## Step 3 — Pick one city to test (first configured city)

Before looping over all configured cities, let's manually walk through one city so we can
visually inspect each step.

In [ ]:
city = TEST_CITY
city

In [ ]:
clipped = clip_to_city(gdf_metric, city["lat"], city["lon"], config.clip_radius_km, metric_crs=METRIC_CRS)
print(f"{len(clipped)} units within {config.clip_radius_km}km of {city['name']}")

fig, ax = plt.subplots(figsize=(8, 8))
clipped.plot(ax=ax, column="pop_density", legend=True, cmap="viridis")
ax.set_title(f"Clipped units around {city['name']} (coloured by density)")
plt.show()

## Step 4 — Filter to dense units

In [ ]:
density_threshold = config.filters["density_threshold"]
dense = filter_dense_units(clipped, density_threshold)
print(f"{len(dense)} units above {density_threshold} people/km²")

fig, ax = plt.subplots(figsize=(8, 8))
clipped.plot(ax=ax, color="lightgrey", edgecolor="white")
dense.plot(ax=ax, color="orangered")
ax.set_title(f"Dense units (>{density_threshold}/km²) highlighted")
plt.show()

## Step 5 — Buffer + dissolve into clusters

In [ ]:
dissolved = dissolve_clusters(dense, buffer_m=500)
print(f"{len(dissolved)} contiguous cluster(s) found")

fig, ax = plt.subplots(figsize=(8, 8))
clipped.plot(ax=ax, color="lightgrey", edgecolor="white")
dissolved.plot(ax=ax, color="orangered", alpha=0.6, edgecolor="black")
ax.set_title("Dissolved clusters (after 500m buffer)")
plt.show()

## Step 6 — Summarise population/area per cluster

In [ ]:
summarised = summarise_clusters(dense, dissolved)
summarised[["cluster_id", "population", "area_km2", "pop_density"]]

## Step 7 — Apply city-level filters

In [ ]:
filtered = apply_city_filters(
    summarised,
    min_population=config.filters["min_population"],
    min_area_km2=config.filters["min_area_km2"],
)
filtered[["cluster_id", "population", "area_km2", "pop_density", "passes_filter"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
clipped.plot(ax=ax, color="lightgrey", edgecolor="white")
filtered[filtered["passes_filter"]].plot(ax=ax, color="green", alpha=0.6, edgecolor="black")
filtered[~filtered["passes_filter"]].plot(ax=ax, color="red", alpha=0.4, edgecolor="black")
ax.set_title(f"{city['name']} — green = passes filter, red = fails")
plt.show()

## Step 8 — Run all configured cities using `process_city`

Now that we've manually verified each step works for the first configured city, run the
full pipeline across every city in the config.

In [ ]:
results = []
for city in config.cities:
    result = process_city(
        gdf_metric,
        city,
        clip_radius_km=config.clip_radius_km,
        density_threshold=config.filters["density_threshold"],
        min_population=config.filters["min_population"],
        min_area_km2=config.filters["min_area_km2"],
        metric_crs=METRIC_CRS,
    )
    results.append(result)

    clusters = result["clusters"]
    if clusters is None:
        print(f"{result['name']:12s} -> {result['status']}")
        continue

    passing = clusters[clusters["passes_filter"]]
    if not passing.empty:
        top = passing.sort_values("population", ascending=False).iloc[0]
        print(
            f"{result['name']:12s} -> PASS   "
            f"pop={top['population']:,.0f}  area={top['area_km2']:,.1f}km²  "
            f"density={top['pop_density']:,.0f}/km²"
        )
    else:
        best = clusters.sort_values("population", ascending=False).iloc[0]
        print(
            f"{result['name']:12s} -> FAIL   "
            f"best cluster pop={best['population']:,.0f}  "
            f"area={best['area_km2']:,.1f}km²"
        )

## Step 9 — Visual grid of all configured cities

Quick visual sanity check across every city at once.

In [ ]:
import math

n_cities = len(results)
n_cols = min(4, n_cities)
n_rows = math.ceil(n_cities / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = axes.flatten()

for ax, result in zip(axes, results):
    clusters = result["clusters"]
    if clusters is None:
        ax.set_title(f"{result['name']} ({result['status']})")
        ax.axis("off")
        continue

    clusters.plot(ax=ax, color="lightgrey", edgecolor="grey")
    clusters[clusters["passes_filter"]].plot(ax=ax, color="green", alpha=0.7)
    clusters[~clusters["passes_filter"]].plot(ax=ax, color="red", alpha=0.4)
    ax.set_title(result["name"])
    ax.set_xticks([])
    ax.set_yticks([])

for ax in axes[len(results):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Step 10 — Save passing city polygons

Save only the passing clusters (one polygon per city) to `data/processed/`
for use in the next notebook (amenity collection).

In [ ]:
import pandas as pd

output_rows = []
for result in results:
    clusters = result["clusters"]
    if clusters is None:
        continue
    passing = clusters[clusters["passes_filter"]]
    if passing.empty:
        continue
    # Take the largest passing cluster as the city's urban extent
    top = passing.sort_values("population", ascending=False).iloc[[0]].copy()
    top["city"] = result["name"]
    top["city_ascii"] = result.get("name_ascii", normalize_german_city_name(result["name"]))
    top["city_slug"] = result.get("city_slug", city_slug(result["name"]))
    output_rows.append(top)

final_gdf = gpd.GeoDataFrame(pd.concat(output_rows, ignore_index=True), crs=gdf_metric.crs)
final_gdf = final_gdf[
    ["city", "city_ascii", "city_slug", "population", "area_km2", "pop_density", "geometry"]
]
final_gdf

In [ ]:
output_dir = PROJECT_ROOT / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"{COUNTRY}_cities.gpkg"
final_gdf.to_file(output_path, driver="GPKG")
print(f"Saved {len(final_gdf)} city polygons to {output_path}")

## Filter log

Log of every city and whether it passed, for the methodology section.

In [ ]:
log_rows = []
for result in results:
    clusters = result["clusters"]
    city_ascii = result.get("name_ascii", normalize_german_city_name(result["name"]))
    slug = result.get("city_slug", city_slug(result["name"]))
    if clusters is None:
        log_rows.append({"city": result["name"], "city_ascii": city_ascii, "city_slug": slug,
                          "status": result["status"], "population": None, "area_km2": None})
        continue
    passing = clusters[clusters["passes_filter"]]
    if not passing.empty:
        top = passing.sort_values("population", ascending=False).iloc[0]
        log_rows.append({"city": result["name"], "city_ascii": city_ascii, "city_slug": slug,
                          "status": "passed",
                          "population": top["population"], "area_km2": top["area_km2"]})
    else:
        best = clusters.sort_values("population", ascending=False).iloc[0]
        log_rows.append({"city": result["name"], "city_ascii": city_ascii, "city_slug": slug,
                          "status": "filtered_out",
                          "population": best["population"], "area_km2": best["area_km2"]})

log_df = pd.DataFrame(log_rows)
log_df.to_csv(output_dir / f"{COUNTRY}_filter_log.csv", index=False)
log_df